In [ ]:
# Here I am "monkey patching" torch.nn to include my own RMSNorm before anything uses it.
# This patching fixes an error comming from a synthcity dependency (opacus/grad_sample/rms_norm.py)
# when we try to load a new plugin.
# The problem is that the opacus library assumes that my environment has torch.nn.RMSNorm, but my
# PyTorch version is older than 2.3, so RMSNorm doesn't exist.

import torch.nn as nn

if not hasattr(nn, "RMSNorm"):
    class RMSNorm(nn.Module):
        def __init__(self, dim, eps=1e-8):
            super().__init__()
            self.eps = eps
            self.scale = nn.Parameter(torch.ones(dim))

        def forward(self, x):
            norm = x.norm(2, dim=-1, keepdim=True)
            rms = norm / (x.shape[-1] ** 0.5)
            return self.scale * x / (rms + self.eps)
    
    nn.RMSNorm = RMSNorm  # patch torch.nn

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
import pyarrow.feather as feather

from synthcity.plugins import Plugins
from synthcity.plugins.core.dataloader import GenericDataLoader


In [ ]:
# source the smote method synthcity plugin
code_path = ''
file_path = os.path.join(code_path, 'synthcity_plugin_for_smotenc_generator.py')
with open(os.path.expanduser(file_path)) as file:
    exec(file.read())

In [ ]:
from synthcity.plugins import Plugins

generators = Plugins()

generators.add("smotenc", SmoteNCPlugin)

In [ ]:

def build_all_synthetics_synthcity(
    X: pd.DataFrame,
    split_seeds,
    plugin_name: str,                    # e.g. "tvae", "ctgan", "arf", "ddpm", ...
    plugin_kwargs: dict | None = None,
    *,
    task_id: int = 0,
    ds_name: str = "",
    na_strategy: str = "drop",           # "drop" | "ignore"
    set_split_seed: bool = True,         # inject random_state=seed if not provided
):
    """
    For each seed:
      - 50/50 split (use the 'orig' half)
      - handle NAs per `na_strategy`
      - fit chosen synthcity plugin: Plugins().get(plugin_name, **plugin_kwargs)
      - generate synthetic rows and add metadata
    Returns (syn_long, failures)
    """
    all_chunks: list[pd.DataFrame] = []
    failures: list[dict] = []

    for j, seed in enumerate(split_seeds, start=1):
        try:
            # Split
            X_orig, _ = train_test_split(X, test_size=0.5, random_state=seed)

            # NA handling
            if na_strategy == "drop":
                X_proc = X_orig.dropna().reset_index(drop=True)
            elif na_strategy == "ignore":
                X_proc = X_orig.reset_index(drop=True)
            else:
                raise ValueError(f"Unknown na_strategy: {na_strategy}")

            if X_proc.empty:
                failures.append({
                    "task_id": task_id, "split": j, "role": "syn",
                    "error": "empty_after_na_handling",
                })
                continue

            # Data loader (unconditional generation on all columns)
            loader = GenericDataLoader(X_proc)

            # Build the plugin with user-specified kwargs
            params = dict(plugin_kwargs or {})
            if set_split_seed and "random_state" not in params:
                params["random_state"] = seed  # per-split reproducibility

            model = Plugins().get(plugin_name, **params)

            # Fit & generate
            model.fit(loader)
            n = len(loader)
            X_syn = model.generate(count=n).dataframe()

            # Add metadata
            chunk = X_syn.copy()
            chunk.insert(0, "__role__", "syn")
            chunk.insert(0, "__split__", np.int32(j))
            chunk.insert(0, "__task_id__", np.int32(task_id))
            chunk.insert(0, "__dataset__", str(ds_name))

            all_chunks.append(chunk)

        except Exception as e:
            failures.append({
                "task_id": task_id, "split": j, "role": "syn", "error": repr(e),
            })

    syn_long = (
        pd.concat(all_chunks, axis=0, ignore_index=True)
        if all_chunks else
        pd.DataFrame(columns=["__dataset__", "__task_id__", "__split__", "__role__"])
    )
    return syn_long, failures



def enforce_dtypes(dat, 
                   num_variables, 
                   cat_variables):
    """
    Enforce "float64" type for numeric variables and "object" type for the
    categorical variables
    Parameters:
        dat (pd.DataFrame): Input data matrix (numeric, categorical, or mixed).
        num_variables (list): Indices of numeric variables.
        cat_variables (list): Indices of categorical variables.

    Returns:
    pd.DataFrame: with transformed data types
    """
    if num_variables is not None and cat_variables is None:
        dat_N = pd.DataFrame(dat.iloc[:, num_variables], dtype = "float64")
        dat = dat_N

    elif num_variables is None and cat_variables is not None:
        dat_C = pd.DataFrame(dat.iloc[:, cat_variables], dtype = "str")
        dat = dat_C

    elif num_variables is not None and cat_variables is not None:
        dat_N = pd.DataFrame(dat.iloc[:, num_variables], dtype = "float64")
        dat_C = pd.DataFrame(dat.iloc[:, cat_variables], dtype = "str")
        dat = pd.concat([dat_N, dat_C], axis=1)
        # Reorder columns to match the order in the original data
        reordered_indices = num_variables + cat_variables
        dat = dat.iloc[:, np.argsort(reordered_indices)]

    else:
        raise ValueError("At least one of num_variables or cat_variables must be specified.")
    
    return dat 


def synth_smotenc(
    dat: pd.DataFrame,
    k: int,
    round_int_vars: bool = True
) -> pd.DataFrame:
    """
    Python translation of SynthSMOTENC.

    Implements a simple SMOTE-like synthesizer:
      - k-NN is computed in the space of numeric columns only.
      - Numeric columns are synthesized by interpolation with one randomly
        chosen neighbor among the k neighbors.
      - Categorical / boolean / string columns are synthesized by majority vote
        across {sample} U {its k numeric-nearest neighbors}.

    Parameters
    ----------
    dat : pd.DataFrame
        Input data.
    k : int
        Number of nearest neighbors.
    round_int_vars : bool, default True
        If True, round back columns that are integer-valued in `dat`.
    atol : float, default 1e-8
        Tolerance to decide if an original numeric column is integer-like.

    Returns
    -------
    pd.DataFrame
        Synthetic data with the same shape/columns/index as `dat`.
    """

    # ---- Identify numeric vs. categorical-like columns (like GetVariableTypes) ----
    num_cols = dat.select_dtypes(include="number").columns.tolist()
    cat_cols = dat.select_dtypes(include=["category", "object", "string", "bool", "boolean"]).columns.tolist()

    n = len(dat)
    if len(num_cols) == 0 and len(cat_cols) == 0:
        # Nothing to do
        return dat.copy()

    # ---- kNN on numeric columns (if any) ----
    if len(num_cols) > 0:
        X_num = dat[num_cols].to_numpy(dtype=float)
        # sklearn kneighbors on the same data includes self as the first neighbor;
        # we ask for k+1 and then drop the self neighbor to match R's FNN::get.knn behavior.
        nbrs = NearestNeighbors(n_neighbors=min(k + 1, max(1, n)), algorithm="auto")
        nbrs.fit(X_num)
        distances, indices = nbrs.kneighbors(X_num, return_distance=True)
        # Drop self (first column) if present
        if indices.shape[1] > 1:
            nn_index = indices[:, 1:]  # shape: (n, k) when n > 1 and k >= 1
        else:
            # Degenerate case: only one neighbor (itself)
            nn_index = np.zeros((n, 0), dtype=int)
    else:
        nn_index = np.zeros((n, 0), dtype=int)

    # ---- Helpers ----
    def generate_synth_num_data(dat_df: pd.DataFrame, nn_idx: np.ndarray, num_variables: list[str]) -> pd.DataFrame:
        """Interpolate numeric columns toward a random neighbor among k."""
        if len(num_variables) == 0:
            return dat_df.copy()

        out = dat_df.copy()
        Xn = out[num_variables].to_numpy(dtype=float)
        for i in range(n):
            if nn_idx.shape[1] == 0:  # no neighbors
                continue
            # pick one neighbor index uniformly at random
            j_idx = np.random.randint(nn_idx.shape[1])
            nb = nn_idx[i, j_idx]
            lam = np.random.rand()
            # x + lambda * (x_nn - x)
            Xn[i, :] = Xn[i, :] + lam * (Xn[nb, :] - Xn[i, :])
        out[num_variables] = Xn
        return out

    def majority_vote(values: list[str]) -> str:
        """Majority vote with a deterministic tie-break (lexicographic)."""
        s = pd.Series(values, dtype="string")
        counts = s.value_counts(dropna=False)
        # tie-break by label lexicographic to mimic R's sort(table(...), decreasing=TRUE)
        top_count = counts.max()
        candidates = sorted([idx for idx, c in counts.items() if c == top_count], key=lambda x: str(x))
        return candidates[0]

    def generate_synth_cat_data(dat_df: pd.DataFrame, nn_idx: np.ndarray, cat_variables: list[str]) -> pd.DataFrame:
        """Majority vote across sample + its k neighbors for each categorical column."""
        if len(cat_variables) == 0:
            return dat_df.copy()

        out = dat_df.copy()
        for i in range(n):
            # neighbor indices (could be empty if no numeric columns)
            neigh = nn_idx[i, :].tolist() if nn_idx.shape[1] else []
            # for each categorical column, vote among sample + neighbors
            for col in cat_variables:
                vals = [dat_df.iloc[i][col]]
                if neigh:
                    vals.extend(dat_df.iloc[neigh][col].astype("string").tolist())
                out.at[out.index[i], col] = majority_vote(vals)
        return out

    # ---- Generate numeric-only and categorical-only synthetic datasets ----
    synth_num_dat = generate_synth_num_data(dat, nn_index, num_cols)
    synth_cat_dat = generate_synth_cat_data(dat, nn_index, cat_cols)

    # ---- Combine: numeric from synth_num_dat + categorical from synth_cat_dat ----
    synthetic_dat = synth_num_dat.copy()
    if len(cat_cols) > 0:
        synthetic_dat[cat_cols] = synth_cat_dat[cat_cols]

    # ---- Optionally round back integer-valued numeric columns (like RoundIntegerVariables) ----
    if round_int_vars and len(num_cols) > 0:
        synthetic_dat = round_integer_variables(df_ori=dat, df_syn=synthetic_dat)

    # Preserve original column order/dtypes as much as feasible
    synthetic_dat = synthetic_dat.astype(
        {
            c: dat[c].dtype
            for c in synthetic_dat.columns
            if isinstance(dat[c].dtype, pd.CategoricalDtype)
        },
        copy=False,
    )
    
    return synthetic_dat


def round_integer_variables(df_ori: pd.DataFrame, df_syn: pd.DataFrame, atol: float = 1e-8) -> pd.DataFrame:
    """
    This function uses the original data to determine which variables
    are integer-valued and then rounds the values of the corresponding
    variables in the synthetic data to the nearest integer.

    Parameters
    ----------
    df_ori : pd.DataFrame
        DataFrame containing the original data.
    df_syn : pd.DataFrame
        DataFrame containing the synthetic data.
        Assumes same schema/columns as `df_ori`.
    atol : float, optional
        Absolute tolerance when testing whether original data is
        integer-valued (default = 1e-8). This mimics R's all.equal().

    Returns
    -------
    pd.DataFrame
        Synthetic data with rounded values for integer-valued variables.
    """
    # Make sure df_ori and df_syn are pandas dataframes
    df_ori = pd.DataFrame(df_ori)
    df_syn = pd.DataFrame(df_syn)
    
    # Make a copy so original synthetic data is not modified
    syn = df_syn.copy()

    # Iterate only over numeric columns in the original data
    for col in df_ori.select_dtypes(include="number").columns:
        # Drop missing values to avoid issues when checking integer-ness
        s = df_ori[col].dropna()

        # If column is not empty, check if all values are "integer-like"
        # i.e., equal to their rounded version (within tolerance)
        if len(s) and np.isclose(s.to_numpy(), np.round(s.to_numpy()), atol=atol).all():
            # Round the synthetic data for this column to the nearest integer
            #syn[col] = np.round(syn[col])
            syn[col] = np.rint(pd.to_numeric(syn[col], errors="coerce"))

    return syn


In [ ]:
output_path = ''
split_seeds = list(range(1, 11))  # 10 splits

In [ ]:
# ----------------- Abalone data --------------------------------------------------------------

from sklearn.datasets import fetch_openml

# Fetch the Abalone dataset
abalone = fetch_openml(name="abalone", version=1, as_frame=True)

# Access the data and target
X = abalone.data
y = abalone.target

X['target'] =  y # Rings


num_idx = [1, 2, 3, 4, 5, 6, 7, 8]
cat_idx = [0]

X = enforce_dtypes(dat = X, 
                   num_variables = num_idx, 
                   cat_variables = cat_idx)

ds_name = 'abalone'

# ------------------ Generate synthetic datasets using bayesian_network -----------------------------------------
print('running bayesian_network')
syn_long_bayesnet, failures = build_all_synthetics_synthcity(
    X=X,
    split_seeds=split_seeds,
    plugin_name = 'bayesian_network',   
    plugin_kwargs = {'struct_learning_search_method': 'hillclimb',
                        'struct_learning_score': 'bic'},
    task_id=0,
    ds_name=ds_name
)
# Save the combined table once:
fname = os.path.join(output_path, f"{ds_name}_syn_bayesnet.feather")
feather.write_feather(syn_long_bayesnet, fname)
print(len(failures), failures[:1])


# ------------------ Generate synthetic datasets using arf -----------------------------------------
print('running arf')
syn_long_arf, failures = build_all_synthetics_synthcity(
    X=X,
    split_seeds=split_seeds,
    plugin_name = 'arf',   
    plugin_kwargs = {'num_trees': 80,
                        'delta': 0,
                        'max_iters': 2,
                        'early_stop': False,
                        'min_node_size': 2},
    task_id=0,
    ds_name=ds_name
)
# Save the combined table once:
fname = os.path.join(output_path, f"{ds_name}_syn_arf.feather")
feather.write_feather(syn_long_arf, fname)
print(len(failures), failures[:1])


# ------------------ Generate synthetic datasets using ctgan -----------------------------------------
print('running ctgan')
syn_long_ctgan, failures = build_all_synthetics_synthcity(
    X=X,
    split_seeds=split_seeds,
    plugin_name = 'ctgan',   
    plugin_kwargs = {'generator_n_layers_hidden': 1,
                            'generator_n_units_hidden': 100,
                            'generator_nonlin': 'elu',
                            'n_iter': 700,
                            'generator_dropout': 0.13836424598477665,
                            'discriminator_n_layers_hidden': 2,
                            'discriminator_n_units_hidden': 100,
                            'discriminator_nonlin': 'tanh',
                            'discriminator_n_iter': 5,
                            'discriminator_dropout': 0.023861565936528797,
                            'lr': 0.001,
                            'weight_decay': 0.0001,
                            'batch_size': 200,
                            'encoder_max_clusters': 8},
    task_id=0,
    ds_name=ds_name
)
# Save the combined table once:
fname = os.path.join(output_path, f"{ds_name}_syn_ctgan.feather")
feather.write_feather(syn_long_ctgan, fname)
print(len(failures), failures[:1])


# ------------------ Generate synthetic datasets using tvae -----------------------------------------
print('running tvae')
syn_long_tvae, failures = build_all_synthetics_synthcity(
    X=X,
    split_seeds=split_seeds,
    plugin_name = 'tvae',   
    plugin_kwargs = {'n_iter': 400,
                          'lr': 0.001,
                          'decoder_n_layers_hidden': 5,
                          'weight_decay': 0.0001,
                          'batch_size': 128,
                          'n_units_embedding': 200,
                          'decoder_n_units_hidden': 150,
                          'decoder_nonlin': 'tanh',
                          'decoder_dropout': 0.19964446358158816,
                          'encoder_n_layers_hidden': 4,
                          'encoder_n_units_hidden': 100,
                          'encoder_nonlin': 'relu',
                          'encoder_dropout': 0.0820245231222064},
    task_id=0,
    ds_name=ds_name
)
# Save the combined table once:
fname = os.path.join(output_path, f"{ds_name}_syn_tvae.feather")
feather.write_feather(syn_long_tvae, fname)
print(len(failures), failures[:1])


# ------------------ Generate synthetic datasets using ddpm -----------------------------------------
print('running ddpm')
syn_long_ddpm, failures = build_all_synthetics_synthcity(
    X=X,
    split_seeds=split_seeds,
    plugin_name = 'ddpm',   
    plugin_kwargs = {'lr': 0.002991978123076162,
                          'batch_size': 970,
                          'num_timesteps': 407,
                          'n_iter': 7605,
                          'is_classification': False},
    task_id=0,
    ds_name=ds_name
)
# Save the combined table once:
fname = os.path.join(output_path, f"{ds_name}_syn_ddpm.feather")
feather.write_feather(syn_long_ddpm, fname)
print(len(failures), failures[:1])


# ------------------ Generate synthetic datasets using smote -----------------------------------------
print('running smote')
syn_long_smote, failures = build_all_synthetics_synthcity(
    X=X,
    split_seeds=split_seeds,
    plugin_name = 'smotenc',   
    plugin_kwargs = {'k': 5},
    task_id=0,
    ds_name=ds_name
)
# Save the combined table once:
fname = os.path.join(output_path, f"{ds_name}_syn_smote.feather")
feather.write_feather(syn_long_smote, fname)
print(len(failures), failures[:1])


In [ ]:
# -------------------- Bank marketing data ------------------------------------------------------

import openml

# bank marketing
dataset = openml.datasets.get_dataset(44126) 

X, y, _, attribute_names = dataset.get_data(target=dataset.default_target_attribute)

X['target'] = y

num_idx = list(range(0, 7))
cat_idx = [7]

X = enforce_dtypes(dat = X, 
                   num_variables = num_idx, 
                   cat_variables = cat_idx)


ds_name = 'bank'

# ------------------ Generate synthetic datasets using bayesian_network -----------------------------------------
print('running bayesian_network')
syn_long_bayesnet, failures = build_all_synthetics_synthcity(
    X=X,
    split_seeds=split_seeds,
    plugin_name = 'bayesian_network',   
    plugin_kwargs = {'struct_learning_search_method': 'hillclimb',
                         'struct_learning_score': 'bic'},
    task_id=0,
    ds_name=ds_name
)
# Save the combined table once:
fname = os.path.join(output_path, f"{ds_name}_syn_bayesnet.feather")
feather.write_feather(syn_long_bayesnet, fname)
print(len(failures), failures[:1])


# ------------------ Generate synthetic datasets using arf -----------------------------------------
print('running arf')
syn_long_arf, failures = build_all_synthetics_synthcity(
    X=X,
    split_seeds=split_seeds,
    plugin_name = 'arf',   
    plugin_kwargs = {},
    task_id=0,
    ds_name=ds_name
)
# Save the combined table once:
fname = os.path.join(output_path, f"{ds_name}_syn_arf.feather")
feather.write_feather(syn_long_arf, fname)
print(len(failures), failures[:1])


# ------------------ Generate synthetic datasets using ctgan -----------------------------------------
print('running ctgan')
syn_long_ctgan, failures = build_all_synthetics_synthcity(
    X=X,
    split_seeds=split_seeds,
    plugin_name = 'ctgan',   
    plugin_kwargs = {'generator_n_layers_hidden': 2,
                            'generator_n_units_hidden': 50,
                            'generator_nonlin': 'tanh',
                            'n_iter': 1000,
                            'generator_dropout': 0.0575,
                            'discriminator_n_layers_hidden': 4,
                            'discriminator_n_units_hidden': 150,
                            'discriminator_nonlin': 'relu'},
    task_id=0,
    ds_name=ds_name
)
# Save the combined table once:
fname = os.path.join(output_path, f"{ds_name}_syn_ctgan.feather")
feather.write_feather(syn_long_ctgan, fname)
print(len(failures), failures[:1])


# ------------------ Generate synthetic datasets using tvae -----------------------------------------
print('running tvae')
syn_long_tvae, failures = build_all_synthetics_synthcity(
    X=X,
    split_seeds=split_seeds,
    plugin_name = 'tvae',   
    plugin_kwargs = {'n_iter': 300,
                          'lr': 0.0002,
                          'decoder_n_layers_hidden': 4,
                          'weight_decay': 0.001,
                          'batch_size': 256,
                          'n_units_embedding': 200,
                          'decoder_n_units_hidden': 300,
                          'decoder_nonlin': 'elu',
                          'decoder_dropout': 0.194325119117226,
                          'encoder_n_layers_hidden': 1,
                          'encoder_n_units_hidden': 450,
                          'encoder_nonlin': 'leaky_relu',
                          'encoder_dropout': 0.04288563703094718},
    task_id=0,
    ds_name=ds_name
)
# Save the combined table once:
fname = os.path.join(output_path, f"{ds_name}_syn_tvae.feather")
feather.write_feather(syn_long_tvae, fname)
print(len(failures), failures[:1])


# ------------------ Generate synthetic datasets using ddpm -----------------------------------------
print('running ddpm')
syn_long_ddpm, failures = build_all_synthetics_synthcity(
    X=X,
    split_seeds=split_seeds,
    plugin_name = 'ddpm',   
    plugin_kwargs = {'lr': 0.0009375080542687667,
                          'batch_size': 2929,
                          'num_timesteps': 998,
                          'n_iter': 1051,
                          'is_classification': True},
    task_id=0,
    ds_name=ds_name
)
# Save the combined table once:
fname = os.path.join(output_path, f"{ds_name}_syn_ddpm.feather")
feather.write_feather(syn_long_ddpm, fname)
print(len(failures), failures[:1])


# ------------------ Generate synthetic datasets using smote -----------------------------------------
print('running smote')
syn_long_smote, failures = build_all_synthetics_synthcity(
    X=X,
    split_seeds=split_seeds,
    plugin_name = 'smotenc',   
    plugin_kwargs = {'k': 5},
    task_id=0,
    ds_name=ds_name
)
# Save the combined table once:
fname = os.path.join(output_path, f"{ds_name}_syn_smote.feather")
feather.write_feather(syn_long_smote, fname)
print(len(failures), failures[:1])


In [ ]:
# ----------------- Credit dataset -------------------------------------------------------------

dataset = openml.datasets.get_dataset(44089) 

X, y, _, attribute_names = dataset.get_data(target=dataset.default_target_attribute)

X['target'] = y

num_idx = list(range(0, 10))
cat_idx = [10]

X = enforce_dtypes(dat = X, 
                   num_variables = num_idx, 
                   cat_variables = cat_idx)


ds_name = 'credit'

# ------------------ Generate synthetic datasets using bayesian_network -----------------------------------------
print('running bayesian_network')
syn_long_bayesnet, failures = build_all_synthetics_synthcity(
    X=X,
    split_seeds=split_seeds,
    plugin_name = 'bayesian_network',   
    plugin_kwargs = {'struct_learning_search_method': 'hillclimb',
                         'struct_learning_score': 'bic'},
    task_id=0,
    ds_name=ds_name
)
# Save the combined table once:
fname = os.path.join(output_path, f"{ds_name}_syn_bayesnet.feather")
feather.write_feather(syn_long_bayesnet, fname)
print(len(failures), failures[:1])


# ------------------ Generate synthetic datasets using arf -----------------------------------------
print('running arf')
syn_long_arf, failures = build_all_synthetics_synthcity(
    X=X,
    split_seeds=split_seeds,
    plugin_name = 'arf',   
    plugin_kwargs = {},
    task_id=0,
    ds_name=ds_name
)
# Save the combined table once:
fname = os.path.join(output_path, f"{ds_name}_syn_arf.feather")
feather.write_feather(syn_long_arf, fname)
print(len(failures), failures[:1])


# ------------------ Generate synthetic datasets using ctgan -----------------------------------------
print('running ctgan')
syn_long_ctgan, failures = build_all_synthetics_synthcity(
    X=X,
    split_seeds=split_seeds,
    plugin_name = 'ctgan',   
    plugin_kwargs = {'generator_n_layers_hidden': 2,
                            'generator_n_units_hidden': 50,
                            'generator_nonlin': 'tanh',
                            'n_iter': 1000,
                            'generator_dropout': 0.0575,
                            'discriminator_n_layers_hidden': 4,
                            'discriminator_n_units_hidden': 150,
                            'discriminator_nonlin': 'relu'},
    task_id=0,
    ds_name=ds_name
)
# Save the combined table once:
fname = os.path.join(output_path, f"{ds_name}_syn_ctgan.feather")
feather.write_feather(syn_long_ctgan, fname)
print(len(failures), failures[:1])


# ------------------ Generate synthetic datasets using tvae -----------------------------------------
print('running tvae')
syn_long_tvae, failures = build_all_synthetics_synthcity(
    X=X,
    split_seeds=split_seeds,
    plugin_name = 'tvae',   
    plugin_kwargs = {'n_iter': 300,
                          'lr': 0.0002,
                          'decoder_n_layers_hidden': 4,
                          'weight_decay': 0.001,
                          'batch_size': 256,
                          'n_units_embedding': 200,
                          'decoder_n_units_hidden': 300,
                          'decoder_nonlin': 'elu',
                          'decoder_dropout': 0.194325119117226,
                          'encoder_n_layers_hidden': 1,
                          'encoder_n_units_hidden': 450,
                          'encoder_nonlin': 'leaky_relu',
                          'encoder_dropout': 0.04288563703094718},
    task_id=0,
    ds_name=ds_name
)
# Save the combined table once:
fname = os.path.join(output_path, f"{ds_name}_syn_tvae.feather")
feather.write_feather(syn_long_tvae, fname)
print(len(failures), failures[:1])


# ------------------ Generate synthetic datasets using ddpm -----------------------------------------
print('running ddpm')
syn_long_ddpm, failures = build_all_synthetics_synthcity(
    X=X,
    split_seeds=split_seeds,
    plugin_name = 'ddpm',   
    plugin_kwargs = {'lr': 0.0009375080542687667,
                          'batch_size': 2929,
                          'num_timesteps': 998,
                          'n_iter': 1051,
                          'is_classification': True},
    task_id=0,
    ds_name=ds_name
)
# Save the combined table once:
fname = os.path.join(output_path, f"{ds_name}_syn_ddpm.feather")
feather.write_feather(syn_long_ddpm, fname)
print(len(failures), failures[:1])


# ------------------ Generate synthetic datasets using smote -----------------------------------------
print('running smote')
syn_long_smote, failures = build_all_synthetics_synthcity(
    X=X,
    split_seeds=split_seeds,
    plugin_name = 'smotenc',   
    plugin_kwargs = {'k': 5},
    task_id=0,
    ds_name=ds_name
)
# Save the combined table once:
fname = os.path.join(output_path, f"{ds_name}_syn_smote.feather")
feather.write_feather(syn_long_smote, fname)
print(len(failures), failures[:1])


In [ ]:
# ----------------- Eye movements dataset ----------------------------------------------

dataset = openml.datasets.get_dataset(44130) 

X, y, _, attribute_names = dataset.get_data(target=dataset.default_target_attribute)

X['target'] = y

num_idx = list(range(0, 20))
cat_idx = [20]

X = enforce_dtypes(dat = X, 
                   num_variables = num_idx, 
                   cat_variables = cat_idx)


ds_name = 'eye'

# ------------------ Generate synthetic datasets using bayesian_network -----------------------------------------
print('running bayesian_network')
syn_long_bayesnet, failures = build_all_synthetics_synthcity(
    X=X,
    split_seeds=split_seeds,
    plugin_name = 'bayesian_network',   
    plugin_kwargs = {'struct_learning_search_method': 'hillclimb',
                         'struct_learning_score': 'bic'},
    task_id=0,
    ds_name=ds_name
)
# Save the combined table once:
fname = os.path.join(output_path, f"{ds_name}_syn_bayesnet.feather")
feather.write_feather(syn_long_bayesnet, fname)
print(len(failures), failures[:1])


# ------------------ Generate synthetic datasets using arf -----------------------------------------
print('running arf')
syn_long_arf, failures = build_all_synthetics_synthcity(
    X=X,
    split_seeds=split_seeds,
    plugin_name = 'arf',   
    plugin_kwargs = {},
    task_id=0,
    ds_name=ds_name
)
# Save the combined table once:
fname = os.path.join(output_path, f"{ds_name}_syn_arf.feather")
feather.write_feather(syn_long_arf, fname)
print(len(failures), failures[:1])


# ------------------ Generate synthetic datasets using ctgan -----------------------------------------
print('running ctgan')
syn_long_ctgan, failures = build_all_synthetics_synthcity(
    X=X,
    split_seeds=split_seeds,
    plugin_name = 'ctgan',   
    plugin_kwargs = {'generator_n_layers_hidden': 2,
                            'generator_n_units_hidden': 50,
                            'generator_nonlin': 'tanh',
                            'n_iter': 1000,
                            'generator_dropout': 0.0575,
                            'discriminator_n_layers_hidden': 4,
                            'discriminator_n_units_hidden': 150,
                            'discriminator_nonlin': 'relu'},
    task_id=0,
    ds_name=ds_name
)
# Save the combined table once:
fname = os.path.join(output_path, f"{ds_name}_syn_ctgan.feather")
feather.write_feather(syn_long_ctgan, fname)
print(len(failures), failures[:1])


# ------------------ Generate synthetic datasets using tvae -----------------------------------------
print('running tvae')
syn_long_tvae, failures = build_all_synthetics_synthcity(
    X=X,
    split_seeds=split_seeds,
    plugin_name = 'tvae',   
    plugin_kwargs = {'n_iter': 300,
                          'lr': 0.0002,
                          'decoder_n_layers_hidden': 4,
                          'weight_decay': 0.001,
                          'batch_size': 256,
                          'n_units_embedding': 200,
                          'decoder_n_units_hidden': 300,
                          'decoder_nonlin': 'elu',
                          'decoder_dropout': 0.194325119117226,
                          'encoder_n_layers_hidden': 1,
                          'encoder_n_units_hidden': 450,
                          'encoder_nonlin': 'leaky_relu',
                          'encoder_dropout': 0.04288563703094718},
    task_id=0,
    ds_name=ds_name
)
# Save the combined table once:
fname = os.path.join(output_path, f"{ds_name}_syn_tvae.feather")
feather.write_feather(syn_long_tvae, fname)
print(len(failures), failures[:1])


# ------------------ Generate synthetic datasets using ddpm -----------------------------------------
print('running ddpm')
syn_long_ddpm, failures = build_all_synthetics_synthcity(
    X=X,
    split_seeds=split_seeds,
    plugin_name = 'ddpm',   
    plugin_kwargs = {'lr': 0.0009375080542687667,
                          'batch_size': 2929,
                          'num_timesteps': 998,
                          'n_iter': 1051,
                          'is_classification': True},
    task_id=0,
    ds_name=ds_name
)
# Save the combined table once:
fname = os.path.join(output_path, f"{ds_name}_syn_ddpm.feather")
feather.write_feather(syn_long_ddpm, fname)
print(len(failures), failures[:1])


# ------------------ Generate synthetic datasets using smote -----------------------------------------
print('running smote')
syn_long_smote, failures = build_all_synthetics_synthcity(
    X=X,
    split_seeds=split_seeds,
    plugin_name = 'smotenc',   
    plugin_kwargs = {'k': 5},
    task_id=0,
    ds_name=ds_name
)
# Save the combined table once:
fname = os.path.join(output_path, f"{ds_name}_syn_smote.feather")
feather.write_feather(syn_long_smote, fname)
print(len(failures), failures[:1])


In [ ]:
# ---------------- House_16H dataset --------------------------------------------------------

dataset = openml.datasets.get_dataset(44123) 

X, y, _, attribute_names = dataset.get_data(target=dataset.default_target_attribute)

X['target'] = y

num_idx = list(range(0, 16))
cat_idx = [16]

X = enforce_dtypes(dat = X, 
                   num_variables = num_idx, 
                   cat_variables = cat_idx)


ds_name = 'house16h'

# ------------------ Generate synthetic datasets using bayesian_network -----------------------------------------
print('running bayesian_network')
syn_long_bayesnet, failures = build_all_synthetics_synthcity(
    X=X,
    split_seeds=split_seeds,
    plugin_name = 'bayesian_network',   
    plugin_kwargs = {'struct_learning_search_method': 'hillclimb',
                         'struct_learning_score': 'bic'},
    task_id=0,
    ds_name=ds_name
)
# Save the combined table once:
fname = os.path.join(output_path, f"{ds_name}_syn_bayesnet.feather")
feather.write_feather(syn_long_bayesnet, fname)
print(len(failures), failures[:1])


# ------------------ Generate synthetic datasets using arf -----------------------------------------
print('running arf')
syn_long_arf, failures = build_all_synthetics_synthcity(
    X=X,
    split_seeds=split_seeds,
    plugin_name = 'arf',   
    plugin_kwargs = {},
    task_id=0,
    ds_name=ds_name
)
# Save the combined table once:
fname = os.path.join(output_path, f"{ds_name}_syn_arf.feather")
feather.write_feather(syn_long_arf, fname)
print(len(failures), failures[:1])


# ------------------ Generate synthetic datasets using ctgan -----------------------------------------
print('running ctgan')
syn_long_ctgan, failures = build_all_synthetics_synthcity(
    X=X,
    split_seeds=split_seeds,
    plugin_name = 'ctgan',   
    plugin_kwargs = {'generator_n_layers_hidden': 2,
                            'generator_n_units_hidden': 50,
                            'generator_nonlin': 'tanh',
                            'n_iter': 1000,
                            'generator_dropout': 0.0575,
                            'discriminator_n_layers_hidden': 4,
                            'discriminator_n_units_hidden': 150,
                            'discriminator_nonlin': 'relu'},
    task_id=0,
    ds_name=ds_name
)
# Save the combined table once:
fname = os.path.join(output_path, f"{ds_name}_syn_ctgan.feather")
feather.write_feather(syn_long_ctgan, fname)
print(len(failures), failures[:1])


# ------------------ Generate synthetic datasets using tvae -----------------------------------------
print('running tvae')
syn_long_tvae, failures = build_all_synthetics_synthcity(
    X=X,
    split_seeds=split_seeds,
    plugin_name = 'tvae',   
    plugin_kwargs = {'n_iter': 300,
                          'lr': 0.0002,
                          'decoder_n_layers_hidden': 4,
                          'weight_decay': 0.001,
                          'batch_size': 256,
                          'n_units_embedding': 200,
                          'decoder_n_units_hidden': 300,
                          'decoder_nonlin': 'elu',
                          'decoder_dropout': 0.194325119117226,
                          'encoder_n_layers_hidden': 1,
                          'encoder_n_units_hidden': 450,
                          'encoder_nonlin': 'leaky_relu',
                          'encoder_dropout': 0.04288563703094718},
    task_id=0,
    ds_name=ds_name
)
# Save the combined table once:
fname = os.path.join(output_path, f"{ds_name}_syn_tvae.feather")
feather.write_feather(syn_long_tvae, fname)
print(len(failures), failures[:1])


# ------------------ Generate synthetic datasets using ddpm -----------------------------------------
print('running ddpm')
syn_long_ddpm, failures = build_all_synthetics_synthcity(
    X=X,
    split_seeds=split_seeds,
    plugin_name = 'ddpm',   
    plugin_kwargs = {'lr': 0.0009375080542687667,
                          'batch_size': 2929,
                          'num_timesteps': 998,
                          'n_iter': 1051,
                          'is_classification': True},
    task_id=0,
    ds_name=ds_name
)
# Save the combined table once:
fname = os.path.join(output_path, f"{ds_name}_syn_ddpm.feather")
feather.write_feather(syn_long_ddpm, fname)
print(len(failures), failures[:1])


# ------------------ Generate synthetic datasets using smote -----------------------------------------
print('running smote')
syn_long_smote, failures = build_all_synthetics_synthcity(
    X=X,
    split_seeds=split_seeds,
    plugin_name = 'smotenc',   
    plugin_kwargs = {'k': 5},
    task_id=0,
    ds_name=ds_name
)
# Save the combined table once:
fname = os.path.join(output_path, f"{ds_name}_syn_smote.feather")
feather.write_feather(syn_long_smote, fname)
print(len(failures), failures[:1])


In [ ]:
# ---------------------- MagicTelescope data ---------------------------------------------------

dataset = openml.datasets.get_dataset(44125) 

X, y, _, attribute_names = dataset.get_data(target=dataset.default_target_attribute)

X['target'] = y

num_idx = list(range(0, 10))
cat_idx = [10]

X = enforce_dtypes(dat = X, 
                   num_variables = num_idx, 
                   cat_variables = cat_idx)


ds_name = 'magic'

# ------------------ Generate synthetic datasets using bayesian_network -----------------------------------------
print('running bayesian_network')
syn_long_bayesnet, failures = build_all_synthetics_synthcity(
    X=X,
    split_seeds=split_seeds,
    plugin_name = 'bayesian_network',   
    plugin_kwargs = {'struct_learning_search_method': 'hillclimb',
                         'struct_learning_score': 'bic'},
    task_id=0,
    ds_name=ds_name
)
# Save the combined table once:
fname = os.path.join(output_path, f"{ds_name}_syn_bayesnet.feather")
feather.write_feather(syn_long_bayesnet, fname)
print(len(failures), failures[:1])


# ------------------ Generate synthetic datasets using arf -----------------------------------------
print('running arf')
syn_long_arf, failures = build_all_synthetics_synthcity(
    X=X,
    split_seeds=split_seeds,
    plugin_name = 'arf',   
    plugin_kwargs = {},
    task_id=0,
    ds_name=ds_name
)
# Save the combined table once:
fname = os.path.join(output_path, f"{ds_name}_syn_arf.feather")
feather.write_feather(syn_long_arf, fname)
print(len(failures), failures[:1])


# ------------------ Generate synthetic datasets using ctgan -----------------------------------------
print('running ctgan')
syn_long_ctgan, failures = build_all_synthetics_synthcity(
    X=X,
    split_seeds=split_seeds,
    plugin_name = 'ctgan',   
    plugin_kwargs = {'generator_n_layers_hidden': 2,
                            'generator_n_units_hidden': 50,
                            'generator_nonlin': 'tanh',
                            'n_iter': 1000,
                            'generator_dropout': 0.0575,
                            'discriminator_n_layers_hidden': 4,
                            'discriminator_n_units_hidden': 150,
                            'discriminator_nonlin': 'relu'},
    task_id=0,
    ds_name=ds_name
)
# Save the combined table once:
fname = os.path.join(output_path, f"{ds_name}_syn_ctgan.feather")
feather.write_feather(syn_long_ctgan, fname)
print(len(failures), failures[:1])


# ------------------ Generate synthetic datasets using tvae -----------------------------------------
print('running tvae')
syn_long_tvae, failures = build_all_synthetics_synthcity(
    X=X,
    split_seeds=split_seeds,
    plugin_name = 'tvae',   
    plugin_kwargs = {'n_iter': 300,
                          'lr': 0.0002,
                          'decoder_n_layers_hidden': 4,
                          'weight_decay': 0.001,
                          'batch_size': 256,
                          'n_units_embedding': 200,
                          'decoder_n_units_hidden': 300,
                          'decoder_nonlin': 'elu',
                          'decoder_dropout': 0.194325119117226,
                          'encoder_n_layers_hidden': 1,
                          'encoder_n_units_hidden': 450,
                          'encoder_nonlin': 'leaky_relu',
                          'encoder_dropout': 0.04288563703094718},
    task_id=0,
    ds_name=ds_name
)
# Save the combined table once:
fname = os.path.join(output_path, f"{ds_name}_syn_tvae.feather")
feather.write_feather(syn_long_tvae, fname)
print(len(failures), failures[:1])


# ------------------ Generate synthetic datasets using ddpm -----------------------------------------
print('running ddpm')
syn_long_ddpm, failures = build_all_synthetics_synthcity(
    X=X,
    split_seeds=split_seeds,
    plugin_name = 'ddpm',   
    plugin_kwargs = {'lr': 0.0009375080542687667,
                          'batch_size': 2929,
                          'num_timesteps': 998,
                          'n_iter': 1051,
                          'is_classification': True},
    task_id=0,
    ds_name=ds_name
)
# Save the combined table once:
fname = os.path.join(output_path, f"{ds_name}_syn_ddpm.feather")
feather.write_feather(syn_long_ddpm, fname)
print(len(failures), failures[:1])


# ------------------ Generate synthetic datasets using smote -----------------------------------------
print('running smote')
syn_long_smote, failures = build_all_synthetics_synthcity(
    X=X,
    split_seeds=split_seeds,
    plugin_name = 'smotenc',   
    plugin_kwargs = {'k': 5},
    task_id=0,
    ds_name=ds_name
)
# Save the combined table once:
fname = os.path.join(output_path, f"{ds_name}_syn_smote.feather")
feather.write_feather(syn_long_smote, fname)
print(len(failures), failures[:1])


In [ ]:
# ---------------- Pol data ----------------------------------------------------------------

dataset = openml.datasets.get_dataset(44122) 

X, y, _, attribute_names = dataset.get_data(target=dataset.default_target_attribute)

X['target'] = y

num_idx = list(range(0, 26))
cat_idx = [26]

X = enforce_dtypes(dat = X, 
                   num_variables = num_idx, 
                   cat_variables = cat_idx)


ds_name = 'pol'

# ------------------ Generate synthetic datasets using bayesian_network -----------------------------------------
print('running bayesian_network')
syn_long_bayesnet, failures = build_all_synthetics_synthcity(
    X=X,
    split_seeds=split_seeds,
    plugin_name = 'bayesian_network',   
    plugin_kwargs = {'struct_learning_search_method': 'hillclimb',
                         'struct_learning_score': 'bic'},
    task_id=0,
    ds_name=ds_name
)
# Save the combined table once:
fname = os.path.join(output_path, f"{ds_name}_syn_bayesnet.feather")
feather.write_feather(syn_long_bayesnet, fname)
print(len(failures), failures[:1])


# ------------------ Generate synthetic datasets using arf -----------------------------------------
print('running arf')
syn_long_arf, failures = build_all_synthetics_synthcity(
    X=X,
    split_seeds=split_seeds,
    plugin_name = 'arf',   
    plugin_kwargs = {},
    task_id=0,
    ds_name=ds_name
)
# Save the combined table once:
fname = os.path.join(output_path, f"{ds_name}_syn_arf.feather")
feather.write_feather(syn_long_arf, fname)
print(len(failures), failures[:1])


# ------------------ Generate synthetic datasets using ctgan -----------------------------------------
print('running ctgan')
syn_long_ctgan, failures = build_all_synthetics_synthcity(
    X=X,
    split_seeds=split_seeds,
    plugin_name = 'ctgan',   
    plugin_kwargs = {'generator_n_layers_hidden': 2,
                            'generator_n_units_hidden': 50,
                            'generator_nonlin': 'tanh',
                            'n_iter': 1000,
                            'generator_dropout': 0.0575,
                            'discriminator_n_layers_hidden': 4,
                            'discriminator_n_units_hidden': 150,
                            'discriminator_nonlin': 'relu'},
    task_id=0,
    ds_name=ds_name
)
# Save the combined table once:
fname = os.path.join(output_path, f"{ds_name}_syn_ctgan.feather")
feather.write_feather(syn_long_ctgan, fname)
print(len(failures), failures[:1])


# ------------------ Generate synthetic datasets using tvae -----------------------------------------
print('running tvae')
syn_long_tvae, failures = build_all_synthetics_synthcity(
    X=X,
    split_seeds=split_seeds,
    plugin_name = 'tvae',   
    plugin_kwargs = {'n_iter': 300,
                          'lr': 0.0002,
                          'decoder_n_layers_hidden': 4,
                          'weight_decay': 0.001,
                          'batch_size': 256,
                          'n_units_embedding': 200,
                          'decoder_n_units_hidden': 300,
                          'decoder_nonlin': 'elu',
                          'decoder_dropout': 0.194325119117226,
                          'encoder_n_layers_hidden': 1,
                          'encoder_n_units_hidden': 450,
                          'encoder_nonlin': 'leaky_relu',
                          'encoder_dropout': 0.04288563703094718},
    task_id=0,
    ds_name=ds_name
)
# Save the combined table once:
fname = os.path.join(output_path, f"{ds_name}_syn_tvae.feather")
feather.write_feather(syn_long_tvae, fname)
print(len(failures), failures[:1])


# ------------------ Generate synthetic datasets using ddpm -----------------------------------------
print('running ddpm')
syn_long_ddpm, failures = build_all_synthetics_synthcity(
    X=X,
    split_seeds=split_seeds,
    plugin_name = 'ddpm',   
    plugin_kwargs = {'lr': 0.0009375080542687667,
                          'batch_size': 2929,
                          'num_timesteps': 998,
                          'n_iter': 1051,
                          'is_classification': True},
    task_id=0,
    ds_name=ds_name
)
# Save the combined table once:
fname = os.path.join(output_path, f"{ds_name}_syn_ddpm.feather")
feather.write_feather(syn_long_ddpm, fname)
print(len(failures), failures[:1])


# ------------------ Generate synthetic datasets using smote -----------------------------------------
print('running smote')
syn_long_smote, failures = build_all_synthetics_synthcity(
    X=X,
    split_seeds=split_seeds,
    plugin_name = 'smotenc',   
    plugin_kwargs = {'k': 5},
    task_id=0,
    ds_name=ds_name
)
# Save the combined table once:
fname = os.path.join(output_path, f"{ds_name}_syn_smote.feather")
feather.write_feather(syn_long_smote, fname)
print(len(failures), failures[:1])
